<a href="https://colab.research.google.com/github/Rimshakalhoro/flyrank-ml-internship-rimsha/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rimshakalhoro/flyrank-ml-internship-rimsha/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### My baseline rule

My baseline uses a simple action score to prioritize pages that may deserve human review.

A page receives points when it shows observable signs that may indicate a review opportunity:

- Declining performance.
- Meaningful search visibility.
- Lower engagement.
- Older or less fresh content.

The points are added together to create a baseline action score. Pages with higher scores are placed higher in the review queue.

This is a simple transparent rule rather than a machine-learning model. It provides a baseline that future approaches can be compared against.

### Reason codes

The rule can produce the following reason codes:

- `DECLINING_TREND` — the observed trend direction is down.
- `HIGH_VISIBILITY` — the page has relatively high impressions.
- `LOW_ENGAGEMENT` — the engagement rate is relatively low.
- `STALE_CONTENT` — the page has not been updated recently or is classified as less fresh.

A page can have more than one reason code. These reason codes explain why the page was placed in the review queue.

In [5]:
# ==========================================
# SECTION 1 — LOAD DATA AND DEFINE RULE
# ==========================================

import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/Rimshakalhoro/flyrank-ml-internship-rimsha/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nBaseline rule signals:")
print("- Declining trend")
print("- Search visibility")
print("- Low engagement")
print("- Content freshness")


Rows: 30000
Columns: 44

Baseline rule signals:
- Declining trend
- Search visibility
- Low engagement
- Content freshness


### Building the ranked queue

I created a transparent baseline action score using four observable signals.

The rule gives the highest weight to an observed declining trend, followed by meaningful search visibility. Low engagement and stale content provide additional signals.

The thresholds for visibility, engagement, and staleness are calculated from the observed dataset using percentiles. This avoids manually inventing arbitrary numeric cutoffs.

All pages are ranked from the highest baseline action score to the lowest, and the resulting queue is written to `work/outputs/baseline_action_score.csv`.

In [6]:
# ==========================================
# SECTION 2 — BUILD THE RANKED QUEUE
# ==========================================

import pandas as pd
import os

# Make a copy of the dataset
baseline = df.copy()

# Make sure required numeric columns are numeric
numeric_columns = [
    "impressions_90d",
    "engagement_rate",
    "days_since_last_update"
]

for col in numeric_columns:
    baseline[col] = pd.to_numeric(baseline[col], errors="coerce")

# Start with score 0
baseline["action_score"] = 0

# Create an empty reason code column
baseline["reason_codes"] = ""

# ------------------------------------------
# RULE 1 — Declining trend
# +3 points
# ------------------------------------------

declining = (
    baseline["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
)

baseline.loc[declining, "action_score"] += 3
baseline.loc[declining, "reason_codes"] += "DECLINING_TREND; "

# ------------------------------------------
# RULE 2 — High visibility
# Top 25% impressions
# +2 points
# ------------------------------------------

impression_threshold = baseline["impressions_90d"].quantile(0.75)

high_visibility = (
    baseline["impressions_90d"] >= impression_threshold
)

baseline.loc[high_visibility, "action_score"] += 2
baseline.loc[high_visibility, "reason_codes"] += "HIGH_VISIBILITY; "

# ------------------------------------------
# RULE 3 — Low engagement
# Bottom 25% engagement rate
# +1 point
# ------------------------------------------

engagement_threshold = baseline["engagement_rate"].quantile(0.25)

low_engagement = (
    baseline["engagement_rate"].notna()
    & (baseline["engagement_rate"] <= engagement_threshold)
)

baseline.loc[low_engagement, "action_score"] += 1
baseline.loc[low_engagement, "reason_codes"] += "LOW_ENGAGEMENT; "

# ------------------------------------------
# RULE 4 — Stale content
# Top 25% days since update
# +1 point
# ------------------------------------------

stale_threshold = baseline["days_since_last_update"].quantile(0.75)

stale_content = (
    baseline["days_since_last_update"].notna()
    & (baseline["days_since_last_update"] >= stale_threshold)
)

baseline.loc[stale_content, "action_score"] += 1
baseline.loc[stale_content, "reason_codes"] += "STALE_CONTENT; "

# Clean reason codes
baseline["reason_codes"] = (
    baseline["reason_codes"]
    .str.rstrip("; ")
)

# ------------------------------------------
# RANK THE PAGES
# ------------------------------------------

baseline = baseline.sort_values(
    by=["action_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

# ------------------------------------------
# SAVE THE CSV
# ------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline.to_csv(output_path, index=False)

# ------------------------------------------
# SHOW RESULTS
# ------------------------------------------

print("SUCCESS! Baseline ranked queue created.")

print("\nThresholds used:")
print("High visibility (75th percentile):", impression_threshold)
print("Low engagement (25th percentile):", engagement_threshold)
print("Stale content (75th percentile):", stale_threshold)

print("\nScore distribution:")
print(baseline["action_score"].value_counts().sort_index())

print("\nFile saved to:")
print(output_path)

print("\nTOP 10 PAGES:")

baseline[
    [
        "rank",
        "content_id",
        "action_score",
        "reason_codes",
        "impressions_90d",
        "engagement_rate",
        "days_since_last_update"
    ]
].head(10)


SUCCESS! Baseline ranked queue created.

Thresholds used:
High visibility (75th percentile): 3615.25
Low engagement (25th percentile): 0.0
Stale content (75th percentile): 104.0

Score distribution:
action_score
0    1048
1    7667
2    3098
3    2743
4    8169
5    4501
6    2185
7     589
Name: count, dtype: int64

File saved to:
work/outputs/baseline_action_score.csv

TOP 10 PAGES:


,rank,content_id,action_score,reason_codes,impressions_90d,engagement_rate,days_since_last_update
0,1,content_c8e9d6ab9013,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,208678,0.0,104
1,2,content_11fcfd65d94c,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,149083,0.0,104
2,3,content_fb4bf6555c79,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,84093,0.0,104
3,4,content_fea6a0d13b4a,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,79965,0.0,104
4,5,content_94058fab0b5b,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,54754,0.0,104
5,6,content_5daaf566b54c,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,50013,0.0,104
6,7,content_eaebd42b43cd,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,49350,0.0,104
7,8,content_5096a9d25fe5,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,49158,0.0,104
8,9,content_6e28a04c07a8,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,41226,0.0,104
9,10,content_50a9f9f861c6,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,41038,0.0,104


### Top-20 review

I reviewed the top 20 pages produced by the baseline score.

For each recommendation, I record a suggested action, the observable reason codes that caused the page to rank highly, a confidence note, and a condition that could make the recommendation wrong.

The suggested action is not automatic. The ranked queue is intended to help a human decide which pages deserve review first.

In [7]:
# ==========================================
# SECTION 3 — TOP-20 REVIEW
# ==========================================

top20 = baseline.head(20).copy()

def suggest_action(reasons):
    if "DECLINING_TREND" in reasons:
        return "REVIEW_FOR_REFRESH"
    elif "STALE_CONTENT" in reasons:
        return "REVIEW_FRESHNESS"
    elif "LOW_ENGAGEMENT" in reasons:
        return "REVIEW_ENGAGEMENT"
    else:
        return "MONITOR"

def confidence_note(score):
    if score >= 6:
        return "Higher confidence: multiple observable signals support review."
    elif score >= 4:
        return "Medium confidence: several signals support review."
    else:
        return "Lower confidence: limited evidence from the baseline rule."

def what_makes_it_wrong(reasons):
    return (
        "The observed signals may not reflect a problem that content changes can solve. "
        "Performance may be affected by seasonality, demand changes, measurement limits, "
        "or other factors not represented by this baseline rule."
    )

top20["suggested_action"] = top20[
    "reason_codes"
].apply(suggest_action)

top20["confidence_note"] = top20[
    "action_score"
].apply(confidence_note)

top20["what_would_make_it_wrong"] = top20[
    "reason_codes"
].apply(what_makes_it_wrong)

review_columns = [
    "rank",
    "content_id",
    "action_score",
    "reason_codes",
    "suggested_action",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_columns]

top20_review


,rank,content_id,action_score,reason_codes,suggested_action,confidence_note,what_would_make_it_wrong
0,1,content_c8e9d6ab9013,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,REVIEW_FOR_REFRESH,Higher confidence: multiple observable signals...,The observed signals may not reflect a problem...
1,2,content_11fcfd65d94c,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,REVIEW_FOR_REFRESH,Higher confidence: multiple observable signals...,The observed signals may not reflect a problem...
2,3,content_fb4bf6555c79,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,REVIEW_FOR_REFRESH,Higher confidence: multiple observable signals...,The observed signals may not reflect a problem...
3,4,content_fea6a0d13b4a,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,REVIEW_FOR_REFRESH,Higher confidence: multiple observable signals...,The observed signals may not reflect a problem...
4,5,content_94058fab0b5b,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,REVIEW_FOR_REFRESH,Higher confidence: multiple observable signals...,The observed signals may not reflect a problem...
5,6,content_5daaf566b54c,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,REVIEW_FOR_REFRESH,Higher confidence: multiple observable signals...,The observed signals may not reflect a problem...
6,7,content_eaebd42b43cd,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,REVIEW_FOR_REFRESH,Higher confidence: multiple observable signals...,The observed signals may not reflect a problem...
7,8,content_5096a9d25fe5,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,REVIEW_FOR_REFRESH,Higher confidence: multiple observable signals...,The observed signals may not reflect a problem...
8,9,content_6e28a04c07a8,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,REVIEW_FOR_REFRESH,Higher confidence: multiple observable signals...,The observed signals may not reflect a problem...
9,10,content_50a9f9f861c6,7,DECLINING_TREND; HIGH_VISIBILITY; LOW_ENGAGEME...,REVIEW_FOR_REFRESH,Higher confidence: multiple observable signals...,The observed signals may not reflect a problem...


### Weak picks

Some high-ranked pages may still be weak recommendations.

For example, a page can receive a high score because it has high visibility and stale content, but this does not necessarily mean that changing the content is the best action. The page may be affected by seasonality, changing search demand, or factors outside the available dataset.

The baseline rule also uses fixed weights and percentile thresholds, so it may miss useful combinations of signals or prioritize pages for the wrong reason.

### Leakage check

The baseline score uses only observable page-level signals available in the starter dataset.

I did not use product flags, hidden outcome labels, or explicitly future-looking windows as direct scoring features.

However, timing must be treated carefully in future modelling. A final predictive system should define a clear decision point and ensure that all features are available before the outcome being predicted.

In [8]:
# ==========================================
# SECTION 4 — WEAK PICKS AND LEAKAGE CHECK
# ==========================================

# Find pages in the top 20 with fewer reasons
top20["number_of_reasons"] = (
    top20["reason_codes"]
    .str.count(";")
    .add(1)
)

# A score based on fewer signals can be considered
# a candidate for closer review
weak_picks = top20[
    top20["action_score"] <= top20["action_score"].median()
]

print("TOP 20 SCORE SUMMARY:")
print(top20["action_score"].describe())

print("\nPOSSIBLE WEAKER PICKS:")
print(
    weak_picks[
        [
            "rank",
            "content_id",
            "action_score",
            "reason_codes"
        ]
    ]
)

# ------------------------------------------
# LEAKAGE CHECK
# ------------------------------------------

used_for_score = [
    "trend_direction",
    "impressions_90d",
    "engagement_rate",
    "days_since_last_update"
]

print("\nFEATURES USED FOR THE BASELINE SCORE:")
for feature in used_for_score:
    print("-", feature)

print("\nLEAKAGE CHECK:")
print("No client names, URLs, or private queries are used.")
print("No product flags are used.")
print("No explicitly future-window fields are used as scoring features.")

print("\nImportant note:")
print(
    "trend_direction is an observed directional signal and should be "
    "re-evaluated carefully if a future predictive target is defined later."
)


TOP 20 SCORE SUMMARY:
count    20.0
mean      7.0
std       0.0
min       7.0
25%       7.0
50%       7.0
75%       7.0
max       7.0
Name: action_score, dtype: float64

POSSIBLE WEAKER PICKS:
    rank            content_id  action_score  \
0      1  content_c8e9d6ab9013             7   
1      2  content_11fcfd65d94c             7   
2      3  content_fb4bf6555c79             7   
3      4  content_fea6a0d13b4a             7   
4      5  content_94058fab0b5b             7   
5      6  content_5daaf566b54c             7   
6      7  content_eaebd42b43cd             7   
7      8  content_5096a9d25fe5             7   
8      9  content_6e28a04c07a8             7   
9     10  content_50a9f9f861c6             7   
10    11  content_ffe17f71eda2             7   
11    12  content_b9c64974f6cb             7   
12    13  content_d17baf9465a0             7   
13    14  content_e6955a2c59dc             7   
14    15  content_d1c23751bc5e             7   
15    16  content_d8a36f6f9744         

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.